# AKC Unified Visualizer

This notebook runs the same charts as `run_artifacts/visualize.py`:
- pair/track benchmark charts
- compare-style diagnostics
- MonSTER hyperparameter grid chart

In [5]:
from pathlib import Path
import importlib.util
import json
import sys

cwd = Path.cwd().resolve()
if (cwd / 'run_artifacts' / 'visualize.py').exists():
    root = cwd
elif (cwd.parent / 'run_artifacts' / 'visualize.py').exists():
    root = cwd.parent
else:
    raise FileNotFoundError('Could not find run_artifacts/visualize.py from current working directory')

viz_path = root / 'run_artifacts' / 'visualize.py'
spec = importlib.util.spec_from_file_location('akc_visualize', viz_path)
viz = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = viz
spec.loader.exec_module(viz)
print('Loaded module:', viz_path)
print('Project root:', viz.PROJECT_ROOT)

Loaded module: /home/jake/Developer/akc/run_artifacts/visualize.py
Project root: /home/jake/Developer/akc


In [2]:
pair = 'C'
spec = viz.benchmark_v2.PAIRINGS[pair]

monster_cfg = viz._build_monster_config(
    spec=spec,
    theta_base=float(viz.prepare.REFERENCE_THETA_BASE),
    freq_scale=1.0,
    freq_exponent=1.0,
    boost_scale=1.0,
    rotation_scale=1.0,
    axis_mode=None,
    axis_blend=1.0,
    block_mode='lorentz',
)

pair_outputs = viz.run_pair_track_reports(
    results_tsv=viz.PROJECT_ROOT / 'benchmark_results.tsv',
    benchmark_version=None,
    statuses=('keep', 'ablation'),
    out_dir=viz.RUN_ARTIFACTS / 'plots_v2',
)

compare_outputs = viz.run_compare_style_diagnostics(
    spec=spec,
    seq_len=64,
    demo_freq=24,
    theta_base=float(viz.prepare.REFERENCE_THETA_BASE),
    monster_cfg=monster_cfg,
    out_dir=viz.RUN_ARTIFACTS / 'diagnostics',
)

hyper_outputs = viz.run_monster_hyper_grid(
    cfg=viz.HyperGridConfig(),
    out_dir=viz.RUN_ARTIFACTS / 'diagnostics',
)

manifest = {
    'pair': pair,
    'pair_outputs': pair_outputs,
    'compare_outputs': compare_outputs,
    'hyper_outputs': hyper_outputs,
}
print(json.dumps(manifest, indent=2))

{
  "pair": "C",
  "pair_outputs": {
    "manifest": "/home/jake/Developer/akc/run_artifacts/plots_v2/manifest_monster-v2.1.json",
    "score_plot": "/home/jake/Developer/akc/run_artifacts/plots_v2/pair_score_comparison_monster-v2.1.png",
    "rope_metrics": "/home/jake/Developer/akc/run_artifacts/plots_v2/pair_rope_metrics_monster-v2.1.png",
    "time_metrics": "/home/jake/Developer/akc/run_artifacts/plots_v2/pair_time_metrics_monster-v2.1.png",
    "summary": "/home/jake/Developer/akc/run_artifacts/plots_v2/pair_summary_monster-v2.1.md"
  },
  "compare_outputs": {
    "mean_norm": "/home/jake/Developer/akc/run_artifacts/diagnostics/compare_mean_norm_C.png",
    "probe_heatmaps": "/home/jake/Developer/akc/run_artifacts/diagnostics/compare_probe_heatmaps_C.png",
    "similarity_heatmaps": "/home/jake/Developer/akc/run_artifacts/diagnostics/compare_similarity_heatmaps_C.png",
    "offset_curves": "/home/jake/Developer/akc/run_artifacts/diagnostics/compare_offset_curves_C.png",
    "loca

In [3]:
# CLI-equivalent run
# !uv run python run_artifacts/visualize.py --pair C --benchmark-version monster-v2.1